# ATM Cash Forecasting - Example Usage

This notebook demonstrates how to use the ATM Cash Forecasting system.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Add src to path
sys.path.append('../src')

from data_pipeline import ATMDataPipeline
from models.lightgbm_model import LightGBMForecaster, MultiATMLightGBM
from backtesting import ATMBacktester
from explainability import ATMExplainer

%matplotlib inline

## 1. Load Configuration

In [ ]:
# Load config
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(yaml.dump(config, default_flow_style=False))

## 2. Generate and Load Data

In [ ]:
# Initialize pipeline
pipeline = ATMDataPipeline(config)

# Generate sample data
data_path = pipeline.generate_sample_data('../data/raw', n_atms=10, n_days=180)

# Load data
df = pipeline.ingest_csv(data_path)
print(f"Loaded {len(df)} records for {df['atm_id'].nunique()} ATMs")
df.head()

## 3. Data Cleaning and Feature Engineering

In [ ]:
# Clean data
df_clean = pipeline.clean_data(df, method='linear')
print("Data cleaned")

# Engineer features
df_features = pipeline.engineer_features(df_clean, target_col='cash_withdrawn')
print(f"Created {len(df_features.columns)} features")

# Display feature names
print("\nFeatures:")
print(df_features.columns.tolist())

## 4. Visualize Data

In [ ]:
# Plot cash withdrawal trends for a few ATMs
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
atm_ids = df['atm_id'].unique()[:4]

for idx, atm_id in enumerate(atm_ids):
    ax = axes[idx // 2, idx % 2]
    atm_data = df[df['atm_id'] == atm_id].sort_values('date')
    ax.plot(atm_data['date'], atm_data['cash_withdrawn'] / 100000)
    ax.set_title(f'{atm_id} - Cash Withdrawal Trend')
    ax.set_xlabel('Date')
    ax.set_ylabel('Cash Withdrawn (₹ Lakhs)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Train Model

In [ ]:
# Remove rows with NaN
feature_cols = [col for col in df_features.columns 
               if col not in ['date', 'cash_withdrawn', 'atm_id', 'latitude', 'longitude']]
df_features_clean = df_features.dropna(subset=feature_cols + ['cash_withdrawn'])

# Train global model
lgb_config = config.get('models', {}).get('lightgbm', {})
model_wrapper = MultiATMLightGBM(lgb_config, mode='global')
model_wrapper.train(df_features_clean, target_col='cash_withdrawn')

print("\nModel trained successfully!")

## 6. Make Predictions

In [ ]:
# Get latest data for one ATM
test_atm = atm_ids[0]
test_data = df_features_clean[df_features_clean['atm_id'] == test_atm].tail(10)

# Make predictions
X_test = test_data[feature_cols]
predictions = model_wrapper.predict(X_test)
actuals = test_data['cash_withdrawn'].values

# Display results
results_df = pd.DataFrame({
    'Date': test_data['date'].values,
    'Actual (Lakhs)': actuals / 100000,
    'Predicted (Lakhs)': predictions / 100000,
    'Error (Lakhs)': (actuals - predictions) / 100000
})

print(f"Predictions for {test_atm}:")
print(results_df)

## 7. Explainability with SHAP

In [ ]:
# Create explainer
explainer = ATMExplainer(
    model_wrapper.global_model.model,
    feature_cols,
    config.get('explainability', {})
)

# Calculate SHAP values for test data
explainer.calculate_shap_values(X_test)

# Get top features
top_features = explainer.get_top_features(n=10)
print("\nTop 10 Most Important Features:")
print(top_features)

In [ ]:
# Generate explanation for a specific prediction
idx = 0  # First prediction
explanation = explainer.explain_prediction(
    test_atm,
    str(test_data.iloc[idx]['date']),
    explainer.shap_values[idx],
    X_test.iloc[idx],
    predictions[idx]
)

print("\nNatural Language Explanation:")
print(explanation)

In [ ]:
# Plot waterfall chart
fig = explainer.plot_waterfall(
    explainer.shap_values[idx],
    X_test.iloc[idx],
    predictions[idx],
    max_display=8
)
plt.show()

## 8. Feature Importance

In [ ]:
# Get feature importance from model
feature_importance = model_wrapper.global_model.get_feature_importance(top_n=15)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()

## 9. Backtesting

In [ ]:
# Create backtester
backtester = ATMBacktester(config)

# Define training and prediction functions
def train_fn(train_df):
    X = train_df[feature_cols]
    y = train_df['cash_withdrawn']
    model = LightGBMForecaster(lgb_config)
    model.train(X, y)
    return model

def predict_fn(model, test_df):
    X = test_df[feature_cols]
    return model.predict(X)

# Run backtesting
results = backtester.backtest_model(
    train_fn,
    predict_fn,
    df_features_clean,
    target_col='cash_withdrawn',
    n_splits=3,
    test_size=20
)

print("\nBacktesting Results:")
print("Average Metrics:")
for metric, value in results['average_metrics'].items():
    print(f"  {metric}: {value:.2f}")

## Summary

This notebook demonstrated:
1. Data loading and preprocessing
2. Feature engineering
3. Model training
4. Making predictions
5. SHAP-based explainability
6. Feature importance analysis
7. Backtesting

For the full interactive dashboard, run: `streamlit run app.py`